# 07: Advanced Vectorization & Einsum (Exercises 86–100)

Tackle batch einsum matrix products, Conway's Game of Life in NumPy, Cartesian products, bit manipulation, and bootstrap confidence intervals.

---


In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")

### Exercise 86: Sum of p matrix products at once using einsum
**Difficulty:** `★★★`  
**Tags:** `Einsum, Batch Processing`

#### 💡 Intuition & Concept
Einstein summation notation (`np.einsum`) performs contraction across arbitrary dimensions. `'ijk,ikl->jl'` multiplies pairs of matrices along batch axis `i` and sums across batches.

#### ⚠️ Key Takeaway & Gotchas
Far faster and more memory efficient than looping through batches.


In [ ]:
p, n = 5, 3
M = np.ones((p, n, n))
V = np.ones((p, n, 1))
res = np.einsum('ijk,ikl->jl', M, V)
print("Result shape:", res.shape)
print(res)

### Exercise 87: Consider 16x16 array, get block-sum with 4x4 blocks
**Difficulty:** `★★☆`  
**Tags:** `Reshaping, Block Operations`

#### 💡 Intuition & Concept
Reshape `(16, 16)` into 4D tensor `(4, 4, 4, 4)` and sum across sub-axes `(1, 3)` to pool $4\times 4$ blocks.

#### ⚠️ Key Takeaway & Gotchas
This is equivalent to a 2D average pooling / sum pooling layer in deep learning.


In [ ]:
Z = np.ones((16, 16))
block_sum = Z.reshape(4, 4, 4, 4).sum(axis=(1, 3))
print("Block sum shape:", block_sum.shape)
print("Values (4x4=16):\n", block_sum)

### Exercise 88: Implement Conway's Game of Life using numpy arrays
**Difficulty:** `★★★`  
**Tags:** `Cellular Automata, Convolution`

#### 💡 Intuition & Concept
Conway's rules can be fully vectorized by computing the 8-neighbor sum using slice shifts, then applying boolean masks for birth and survival.

#### ⚠️ Key Takeaway & Gotchas
Zero Python element loops needed—the entire simulation step runs vectorized in C.


In [ ]:
def iterate_life(Z):
    # 8-neighbor count
    N = (Z[0:-2, 0:-2] + Z[0:-2, 1:-1] + Z[0:-2, 2:] +
         Z[1:-1, 0:-2]                 + Z[1:-1, 2:] +
         Z[2:  , 0:-2] + Z[2:  , 1:-1] + Z[2:  , 2:])
         
    birth = (N == 3) & (Z[1:-1, 1:-1] == 0)
    survive = ((N == 2) | (N == 3)) & (Z[1:-1, 1:-1] == 1)
    
    Z[...] = 0
    Z[1:-1, 1:-1][birth | survive] = 1
    return Z

grid = np.zeros((6, 6), dtype=int)
# Blinker oscillator pattern:
grid[2, 1:4] = 1
print("Generation 0:\n", grid)
grid = iterate_life(grid)
print("Generation 1 (oscillated):\n", grid)

### Exercise 89: How to get the n largest values of an array?
**Difficulty:** `★★☆`  
**Tags:** `Partition, Algorithms`

#### 💡 Intuition & Concept
`np.argpartition(-Z, n)[:n]` finds the top $n$ indices in $O(N)$ average time without sorting the whole array ($O(N \log N)$).

#### ⚠️ Key Takeaway & Gotchas
Sorting the entire array is wasteful when $N$ is large and $n$ is small.


In [ ]:
rng = np.random.default_rng(42)
Z = rng.permutation(1000)
n = 5
top_n = Z[np.argpartition(-Z, n)[:n]]
# Sort just the top n:
top_n.sort()
print(f"Top {n} largest values:", top_n[::-1])

### Exercise 90: Cartesian product of arbitrary vectors
**Difficulty:** `★★★`  
**Tags:** `Combinatorics, np.indices`

#### 💡 Intuition & Concept
Computes all possible combinations of elements from multiple vectors without nested loops.

#### ⚠️ Key Takeaway & Gotchas
Creates an array of shape `(len(v1)*len(v2)*..., num_vectors)`.


In [ ]:
def cartesian(arrays):
    arrays = [np.asarray(a) for a in arrays]
    shape = [len(x) for x in arrays]
    ix = np.indices(shape, dtype=int).reshape(len(arrays), -1).T
    res = np.empty_like(ix)
    for n, arr in enumerate(arrays):
        res[:, n] = arr[ix[:, n]]
    return res

v1 = np.array([1, 2])
v2 = np.array([10, 20, 30])
print("Cartesian product:\n", cartesian([v1, v2]))

### Exercise 91: Create a record array from a regular array
**Difficulty:** `★☆☆`  
**Tags:** `Record Arrays, np.rec`

#### 💡 Intuition & Concept
`np.rec.fromarrays` or `np.rec.array` converts arrays and dtypes into record arrays, which allow field access by attribute (`R.x` in addition to `R['x']`).

#### ⚠️ Key Takeaway & Gotchas
Convenient for rapid prototyping, though structured arrays have slightly less overhead.


In [ ]:
Z = np.array([(1, 2.5, 'Alpha'), (2, 3.8, 'Beta')], 
             dtype=[('id', int), ('val', float), ('label', 'U10')])
R = np.rec.array(Z)
print("Record field via attribute:", R.val)
print("First record label:        ", R[0].label)

### Exercise 92: Compute large vector Z to the power of 3 using 3 different methods
**Difficulty:** `★★★`  
**Tags:** `Benchmarking, Performance`

#### 💡 Intuition & Concept
Compares `np.power(Z, 3)`, `Z * Z * Z`, and `np.einsum('i,i,i->i', Z, Z, Z)` for speed.

#### ⚠️ Key Takeaway & Gotchas
`Z * Z * Z` is typically fastest because repeated multiplication avoids generic exponential power math routines.


In [ ]:
import time
Z = np.random.default_rng(42).random(1_000_000)

t0 = time.perf_counter()
r1 = np.power(Z, 3)
t1 = time.perf_counter()
r2 = Z * Z * Z
t2 = time.perf_counter()
r3 = np.einsum('i,i,i->i', Z, Z, Z)
t3 = time.perf_counter()

print(f"np.power:  {(t1-t0)*1000:.2f} ms")
print(f"Z * Z * Z: {(t2-t1)*1000:.2f} ms")
print(f"np.einsum: {(t3-t2)*1000:.2f} ms")
assert np.allclose(r1, r2) and np.allclose(r2, r3)

### Exercise 93: Find rows of A that contain elements of each row of B regardless of order
**Difficulty:** `★★★`  
**Tags:** `Broadcasting, Set Matching`

#### 💡 Intuition & Concept
Uses 4D dimensional broadcasting `(A[..., None, None] == B)` to compare all elements across all rows.

#### ⚠️ Key Takeaway & Gotchas
Evaluates complex subset criteria in pure vectorized NumPy.


In [ ]:
A = np.array([
    [1, 2, 3],
    [2, 3, 4],
    [1, 4, 5]
])
B = np.array([
    [2, 3],
    [4, 5]
])

C = (A[..., np.newaxis, np.newaxis] == B)
matching_rows = np.where(C.any((3, 1)).all(1))[0]
print("Matching rows in A:", matching_rows)

### Exercise 94: Extract rows with unequal values from 10x3 matrix
**Difficulty:** `★★☆`  
**Tags:** `Boolean Indexing, Row Filtering`

#### 💡 Intuition & Concept
Compare each column to the first column: `Z[:, 1:] == Z[:, :-1]`. If all columns match (`.all(axis=1)`), the row has identical elements; negating filters for unequal rows.

#### ⚠️ Key Takeaway & Gotchas
Efficiently weeds out homogeneous rows.


In [ ]:
Z = np.array([
    [1, 1, 1],
    [1, 2, 1],
    [3, 3, 3],
    [4, 5, 6]
])
unequal_rows = Z[~np.all(Z[:, 1:] == Z[:, :-1], axis=1)]
print("Filtered rows with unequal values:\n", unequal_rows)

### Exercise 95: Convert a vector of ints into a matrix binary representation
**Difficulty:** `★★☆`  
**Tags:** `Bitwise, np.unpackbits`

#### 💡 Intuition & Concept
`np.unpackbits` extracts individual bit planes from uint8 bytes into an 8-column binary matrix.

#### ⚠️ Key Takeaway & Gotchas
Input array must be uint8. For wider integers, view as uint8 in little-endian order.


In [ ]:
I = np.array([0, 1, 2, 3, 15, 128], dtype=np.uint8)
binary_matrix = np.unpackbits(I[:, np.newaxis], axis=1)
print("Binary matrix (8-bit representation):\n", binary_matrix)

### Exercise 96: Extract unique rows from 2D array
**Difficulty:** `★★☆`  
**Tags:** `Unique, Rows`

#### 💡 Intuition & Concept
`np.unique(Z, axis=0)` identifies unique rows directly in NumPy.

#### ⚠️ Key Takeaway & Gotchas
In older NumPy versions this required custom void-types, but `axis=0` has been natively supported since v1.13.


In [ ]:
Z = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [1, 2, 3],
    [7, 8, 9]
])
uniq = np.unique(Z, axis=0)
print("Unique rows:\n", uniq)

### Exercise 97: Write einsum equivalent to inner, outer, sum, and mul function
**Difficulty:** `★★☆`  
**Tags:** `Einsum, Equivalences`

#### 💡 Intuition & Concept
`np.einsum` unifies fundamental matrix and tensor operations under a single expressive syntax.

#### ⚠️ Key Takeaway & Gotchas
Understanding index notation in einsum unlocks high-performance tensor computing.


In [ ]:
A = np.array([1, 2, 3])
B = np.array([4, 5, 6])

print("Inner: ", np.einsum('i,i->', A, B), "== np.inner:", np.inner(A, B))
print("Outer:\n", np.einsum('i,j->ij', A, B))
print("Sum:   ", np.einsum('i->', A), "== np.sum:", np.sum(A))
print("Mul:   ", np.einsum('i,i->i', A, B), "== A * B:  ", A * B)

### Exercise 98: Equidistant sampling of a 2D curve path
**Difficulty:** `★★★`  
**Tags:** `Interpolation, Parametric Curves`

#### 💡 Intuition & Concept
Compute cumulative arc-length distances along the curve, and interpolate points at uniformly spaced distance intervals.

#### ⚠️ Key Takeaway & Gotchas
Ensures constant speed traversal along arbitrary non-linear trajectories.


In [ ]:
phi = np.linspace(0, 4 * np.pi, 100)
x = phi * np.cos(phi)
y = phi * np.sin(phi)

# Arc length parameterization
dr = np.hypot(np.diff(x), np.diff(y))
d = np.concatenate(([0], np.cumsum(dr)))
total_length = d[-1]
# Equidistant query points:
d_uniform = np.linspace(0, total_length, 10)
x_sampled = np.interp(d_uniform, d, x)
y_sampled = np.interp(d_uniform, d, y)
print("Sampled equidistant (X, Y) points:")
for xi, yi in zip(x_sampled[:4], y_sampled[:4]):
    print(f"({xi:.2f}, {yi:.2f})")

### Exercise 99: Select rows which can be interpreted as draws from multinomial distribution
**Difficulty:** `★★★`  
**Tags:** `Probability, Multinomial`

#### 💡 Intuition & Concept
For a row to be a valid multinomial draw with $n$ degrees: elements must be non-negative integers and their sum must equal $n$.

#### ⚠️ Key Takeaway & Gotchas
Both conditions can be verified with boolean reductions across rows.


In [ ]:
X = np.array([
    [1, 2, 1],  # Sum = 4, all >= 0 -> Valid
    [0, 4, 0],  # Sum = 4, all >= 0 -> Valid
    [2, 2, 1],  # Sum = 5           -> Invalid
    [3, 1, 0]   # Sum = 4, all >= 0 -> Valid
])
n = 4
is_valid = (X >= 0).all(axis=1) & (X.sum(axis=1) == n)
valid_rows = X[is_valid]
print("Valid multinomial rows (sum=4):\n", valid_rows)

### Exercise 100: Compute bootstrapped 95% confidence intervals for the mean of a 1D array
**Difficulty:** `★★★`  
**Tags:** `Statistics, Bootstrapping, Confidence Interval`

#### 💡 Intuition & Concept
Resample $N_{boot}$ times with replacement using random indices `(N_boot, len(X))`. Compute the mean across samples and extract the 2.5th and 97.5th percentiles.

#### ⚠️ Key Takeaway & Gotchas
No parametric normality assumptions required—fully non-parametric and vectorized.


In [ ]:
rng = np.random.default_rng(42)
X = rng.normal(loc=10.0, scale=2.0, size=100)

n_boot = 2000
boot_indices = rng.integers(0, len(X), (n_boot, len(X)))
boot_means = X[boot_indices].mean(axis=1)
ci_lower, ci_upper = np.percentile(boot_means, [2.5, 97.5])

print(f"Sample Mean:     {X.mean():.3f}")
print(f"95% Bootstrap CI: [{ci_lower:.3f}, {ci_upper:.3f}]")